# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR<sup>2</sup> dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Fetch the list of record set @ids from the dataset
record_sets = dataset.record_sets
print(f"Dataset contains {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    Field name: {field.name}")
        print(f"      @id: {field.id}")
        print(f"      Data type: {field.data_type}")
        print(f"      Description: {field.description}")
    print('-' * 60)

# Store the first record set @id as an example for data extraction
if len(record_sets) > 0:
    record_set_id = record_sets[0].id
else:
    record_set_id = None

# Preview a handful of records by @id
if record_set_id:
    print(f"\nSample records from record set '@id': {record_set_id}:\n")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        if i >= 3:
            break
        print(record)

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. All references use the record set and field `@id`s from above.

In [ ]:
# List all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

if record_set_ids:
    print(f"Columns in record set {record_set_ids[0]}: \n{list(dataframes[record_set_ids[0]].columns)}\n")
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll proceed with EDA by selecting a numeric field by its `@id` and demonstrate basic filtering, normalization, and grouping. Please refer to the overview above for the list of field `@id`s.

In [ ]:
#--- Choose record set and numeric field by @id (replace below if needed with your dataset details) ---#
# For this dataset, let’s try to use the first record set
target_record_set_id = record_set_ids[0]
df = dataframes[target_record_set_id].copy()

# Find a numeric field in the DataFrame (by @id, inferred by dtype as 'int' or 'float')
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

if numeric_field_id is None:
    print("No numeric field detected in the record set.")
else:
    print(f"Using numeric field @id: {numeric_field_id}\n")

    # Threshold filter
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
    display(filtered_df.head())
    
    # Normalization (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
         filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Group by a categorical field if available
    group_field_id = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
            group_field_id = col
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        grouped_df = grouped_df.rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})
        print(f"\nGrouped data by field @id: {group_field_id}")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.xlabel(f"{numeric_field_id}")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
    
    if group_field_id:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use `mlcroissant` to discover, extract, and analyze the FAIR² dataset on clinicopathological and molecular features of second primary colorectal cancer in cancer survivors. We loaded the dataset by referencing entities via their `@id`, performed basic filtering and normalization, and visualized distributions for key numeric variables. For more advanced analysis, consult the field descriptions and data schema via their `@id`s and explore domain-specific questions relevant to clinical oncology.